In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/joebeachcapital/57651-spotify-songs/Spotify Million Song Dataset_exported.csv
/kaggle/input/datasets/negardeylami/semantic-song-similarity/preprocessing.py
/kaggle/input/datasets/negardeylami/semantic-song-similarity/create_embeddings.py
/kaggle/input/datasets/negardeylami/semantic-song-similarity/embedding_utils.py
/kaggle/input/datasets/negardeylami/semantic-song-similarity/evaluation.py
/kaggle/input/datasets/negardeylami/semantic-song-similarity/search.py
/kaggle/input/datasets/negardeylami/semantic-song-similarity/song_lookup.py
/kaggle/input/datasets/negardeylami/semantic-song-similarity/dataset/Songs.csv
/kaggle/input/datasets/negardeylami/semantic-song-similarity/evaluation/eval_queries.csv


In [2]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/joebeachcapital/57651-spotify-songs/Spotify Million Song Dataset_exported.csv', nrows=5)
print(df.columns.tolist())

['artist', 'song', 'link', 'text']


In [3]:
import shutil, os
SRC = '/kaggle/input/datasets/negardeylami/semantic-song-similarity'
DST = '/kaggle/working'

for fname in os.listdir(SRC):
    src_path = os.path.join(SRC, fname)
    dst_path = os.path.join(DST, fname)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy(src_path, dst_path)

os.chdir(DST)
print(os.listdir())

['search.py', 'create_embeddings.py', 'song_lookup.py', 'dataset', 'preprocessing.py', 'evaluation.py', 'embedding_utils.py', 'evaluation', '__notebook__.ipynb']


In [4]:
os.makedirs('embeddings/checkpoints', exist_ok=True)

In [5]:
!pip install sentence-transformers rapidfuzz --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.1 MB/s eta 0:00:00


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# Add Python files

In [7]:
%%writefile preprocessing.py
import re

from matplotlib.pyplot import title


def preprocess_lyrics(lyrics):
    lyrics = lyrics.lower()
    # Remove dataset artifacts
    lyrics = re.sub(  r'\d*embed(?:share)?\s*urlcopyembedcopy', ' ',  lyrics  )
    lyrics = re.sub(r'\[.*?\]', ' ', lyrics)   # remove [Chorus], [Verse 1], [Bridge], etc. in brackets
    lyrics = re.sub(r'\b(chorus|verse|bridge|intro|outro)\b\s*\d*', ' ', lyrics)  
    # Remove punctuation
    lyrics = ''.join(char for char in lyrics if char.isalnum() or char.isspace())
    return re.sub(r'\s+', ' ', lyrics).strip()  # Normalize whitespace

def preprocess_title(title):
    title = title.lower()
    # Remove punctuation
    title = ''.join(char for char in title if char.isalnum() or char.isspace())
    return re.sub(r'\s+', ' ', title).strip()  # Normalize whitespace

def preprocess_query(query):
    query = query.lower()
    # Remove punctuation
    query = ''.join(char for char in query if char.isalnum() or char.isspace())
    return re.sub(r'\s+', ' ', query).strip()  # Normalize whitespace

# punctuation matters for the artist 
def preprocess_artist(artist):
    artist = artist.lower()
    artist = re.sub(r'\s+', ' ', artist).strip()
    return artist

Overwriting preprocessing.py


In [8]:
%%writefile embedding_utils.py
import numpy as np
import os

DEFAULT_ALPHA = 0.95 # ALPHA is the weight we choose for lyrics embeddings
DEFAULT_CHUNK_SIZE = 150
DEFAULT_CHUNK_OVERLAP = 30

def combine_title_lyrics(title_embedding, lyrics_embedding, title_text, alpha = DEFAULT_ALPHA):
    a = 1 if title_text.strip() == '' else alpha
    combined = (1 - a) * title_embedding + a * lyrics_embedding
    return combined / np.linalg.norm(combined)

def combine_title_lyrics_batch(title_embeddings, lyrics_embeddings, title_texts, alpha = DEFAULT_ALPHA):
    is_empty_title = np.array([t.strip() == "" for t in title_texts])
    a = np.where(is_empty_title, 1, alpha)
    combined = (1 - a)[:,None] * title_embeddings + a[:, None] * lyrics_embeddings
    return combined / np.linalg.norm(combined, axis=1, keepdims=True)

def checkpointed_encode(model, texts, checkpoint_path, batch_size = 32):
    emb_path = f"{checkpoint_path}_embeddings.npy"
    progress_path = f"{checkpoint_path}_progress.txt"

    total = len(texts)
    dim = model.get_embedding_dimension()

    if os.path.exists(emb_path) and os.path.exists(progress_path):
        embeddings = np.load(emb_path)
        with open(progress_path) as f:
            done = int(f.read().strip())
        print(f"Resuming from checkpoint: {done}/{total} already encoded...")
    else:
        embeddings = np.zeros((total, dim), dtype=np.float32)
        done = 0

    while done < total:
        batch_end = min(done + batch_size, total)
        batch = texts[done:batch_end]

        batch_embeddings = model.encode(batch, normalize_embeddings = True)
        embeddings[done:batch_end] = batch_embeddings

        done = batch_end

        np.save(emb_path, embeddings)
        with open(progress_path, 'w') as f :
            f.write(str(done))
        if done % (batch_size * 10) == 0 or done == total:
            print(f"Encoded {done}/{total}")

    os.remove(progress_path)
    return embeddings

def chunk_text(text, chunk_size = DEFAULT_CHUNK_SIZE, overlap = DEFAULT_CHUNK_OVERLAP):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk = words[start:start + chunk_size]
        if len(chunk) < 20:
            break
        chunks.append(" ".join(chunk))
    return chunks

def build_chunk_mapping(texts, chunk_size= DEFAULT_CHUNK_SIZE, overlap = DEFAULT_CHUNK_OVERLAP):
    """
    Chunk every song's lyrics and keep track of which song each chunk came from.
    Returns:
      all_chunks: flat list of chunk strings, across all songs
      chunk_song_indices: same length list, chunk_song_indices[i] = which song all_chunks[i] belongs to
    """
    all_chunks = []
    chunk_song_indices = []

    for song_idx, text in enumerate(texts):
        for chunk in chunk_text(text, chunk_size = chunk_size, overlap = overlap ):
            all_chunks.append(chunk)
            chunk_song_indices.append(song_idx)
    return all_chunks, chunk_song_indices


def pool_chunk_embeddings(chunk_embeddings, chunk_song_indices, num_songs):
    """
    Mean-pool chunk-level embeddings back into one embedding per song.
    Vectorized with np.add.at instead of looping per song (matters once
    num_songs is large — a per-song boolean mask loop is O(num_songs * num_chunks)).
    """
    dim = chunk_embeddings.shape[1]
    chunk_song_indices = np.asarray(chunk_song_indices)

    sums = np.zeros((num_songs, dim), dtype=np.float64)
    counts = np.zeros(num_songs, dtype=np.int64)
    np.add.at(sums, chunk_song_indices, chunk_embeddings)
    np.add.at(counts, chunk_song_indices, 1)

    pooled = sums / counts[:, None]
    pooled = pooled / np.linalg.norm(pooled, axis=1, keepdims = True)
    return pooled.astype(np.float32)

def embed_lyrics_single(model, lyrics_cleaned, chunk_size = DEFAULT_CHUNK_SIZE, overlap = DEFAULT_CHUNK_OVERLAP):
    """
    Same chunk + mean-pool logic, for a single song at query time
    (used by search_by_song), so it matches how the database was built.
    """
    chunks = chunk_text(lyrics_cleaned, chunk_size=chunk_size, overlap=overlap)
    chunk_embeddings = model.encode(chunks, normalize_embeddings=True)
    pooled = chunk_embeddings.mean(axis=0)
    return pooled / np.linalg.norm(pooled)


Overwriting embedding_utils.py


In [9]:
%%writefile create_embeddings.py
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from preprocessing import preprocess_lyrics, preprocess_title, preprocess_artist
from embedding_utils import combine_title_lyrics_batch, checkpointed_encode, build_chunk_mapping, pool_chunk_embeddings


data = pd.read_csv('/kaggle/input/datasets/joebeachcapital/57651-spotify-songs/Spotify Million Song Dataset_exported.csv')
data = data.dropna(subset=["text"]).reset_index(drop=True)
# stop_words = set(stopwords.words('english'))
data['Lyrics_cleaned'] = data["text"].apply(preprocess_lyrics)
data['Title_cleaned'] = data["song"].apply(preprocess_title)
data['Artist_cleaned'] = data["artist"].apply(preprocess_artist)

# Sentence Transformer
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
lyrics_list = data['Lyrics_cleaned'].tolist()
title_list = data['Title_cleaned'].fillna('').tolist()

lyrics_chunks, chunk_song_indices = build_chunk_mapping(lyrics_list)


title_embeddings = checkpointed_encode(model, title_list,
checkpoint_path='embeddings/checkpoints/title', batch_size= 32)
# lyrics_embeddings = checkpointed_encode(model, lyrics_list,
# checkpoint_path='embeddings/checkpoints/lyrics', batch_size= 32)

chunk_embeddings = checkpointed_encode(model, lyrics_chunks,
checkpoint_path='embeddings/checkpoints/lyrics', batch_size=32)
lyrics_embeddings = pool_chunk_embeddings(chunk_embeddings, chunk_song_indices, num_songs=len(data))


song_embeddings = combine_title_lyrics_batch(title_embeddings, lyrics_embeddings, data['Title_cleaned'].tolist())
# save the embeddings for later use
np.save('embeddings/song_embeddings.npy', song_embeddings)
np.save('embeddings/title_embeddings.npy' ,title_embeddings)
np.save('embeddings/lyrics_embeddings.npy' ,lyrics_embeddings)


# save metadata
data[["song", "artist", "text", 'Title_cleaned','Artist_cleaned', 'Lyrics_cleaned']].to_csv('embeddings/metadata.csv', index=False)


Overwriting create_embeddings.py


In [10]:
%%writefile song_lookup.py
from rapidfuzz import fuzz, process
from preprocessing import preprocess_title, preprocess_lyrics, preprocess_artist
import pandas as pd

def find_song(data, title, artist=None, threshold=80, top_k = 5):
    title_clean = preprocess_artist(title)
    candidates = data

    # filter by artist if given
    if artist is not None:
        artist_clean = preprocess_artist(artist)
        artist_scores = process.extract(artist_clean, candidates['Artist'].str.lower(),
        scorer = fuzz.WRatio, limit = None)
        good_artist_indices = [idx for _, score, idx in artist_scores if score >= threshold]
        if good_artist_indices:
            candidates = candidates.loc[good_artist_indices]

    # 1. exact match
    exact_matches = candidates[candidates['Title_cleaned'] == title_clean]
    if len(exact_matches) == 1:
        return {"status": "exact", "match": exact_matches.iloc[0]}
    if len(exact_matches) > 1:
        return {"status": "ambiguous", "candidates" : exact_matches.to_dict('records')}

    # 2. no exact match
    results = process.extract(title_clean, candidates['Title_cleaned'],
    scorer=fuzz.WRatio, limit = top_k)

    # results = list of (matched_string, score, index_in_candidates)
    strong_matches = [(candidates.loc[idx], score) for _, score, idx
    in results if score >= threshold]
    if not strong_matches:
        return {"status": "not_found"}
    if len(strong_matches) == 1:
        return {"status":"exact", "match": strong_matches[0][0]}
    return {"status": "fuzzy", "candidates": strong_matches}


# data = pd.read_csv('dataset/Songs.csv')
# data = data.dropna(subset=["Lyrics"]).reset_index(drop=True)
# # stop_words = set(stopwords.words('english'))
# data['Lyrics_cleaned'] = data['Lyrics'].apply(preprocess_lyrics)
# data['Title_cleaned'] = data['Title'].apply(preprocess_title)

# print(find_song(data, 'shape', 'ed 4 sheeran'))

Overwriting song_lookup.py


In [11]:
%%writefile search.py
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from preprocessing import preprocess_query
from song_lookup import find_song
from embedding_utils import combine_title_lyrics, embed_lyrics_single, combine_title_lyrics_batch

# load the embeddings and metadata
title_embeddings = np.load('embeddings/title_embeddings.npy')
lyrics_embeddings = np.load('embeddings/lyrics_embeddings.npy')
data = pd.read_csv('embeddings/metadata.csv')

song_embeddings = combine_title_lyrics_batch(
    title_embeddings, lyrics_embeddings, data['Title_cleaned'].fillna('').tolist(), alpha=1.0
)
song_embeddings = song_embeddings / np.linalg.norm(song_embeddings, axis=1, keepdims=True)
np.save('embeddings/song_embeddings.npy', song_embeddings)

# song_embeddings = np.load('embeddings/song_embeddings.npy')
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def _rank_by_embedding(query_embedding, k=5, exclude_index=None):
    similarity_score = cosine_similarity(query_embedding, song_embeddings).flatten()
    if exclude_index is not None:
        similarity_score[exclude_index] = -1  # keep the queried song out of its own results
    top_indices = np.argsort(similarity_score)[-k:][::-1]
    return [{
        "song": data.iloc[i]["song"],
        "artist": data.iloc[i]["artist"],
        'Similarity': similarity_score[i],
        "text": data.iloc[i]["text"]
        }
        for i in top_indices]

def search_by_text(query_text, k=5):
    query = preprocess_query(query_text)
    query_embedding = model.encode([query], normalize_embeddings=True)
    return { "status": "ok", "results": _rank_by_embedding(query_embedding, k=k)}

def search_by_song(title, artist=None, k=5):
    lookup_result = find_song(data, title, artist = artist)
    if lookup_result["status"] in ("not_found", "ambiguous", "fuzzy"):
        return lookup_result # caller/UI should ask user to disambiguous

    matched_song = lookup_result["match"]
    matched_index = matched_song.name

    title_embedding = model.encode([matched_song["Title_cleaned"]], normalize_embeddings=True)[0]
    lyrics_embedding = embed_lyrics_single(model, matched_song["Lyrics_cleaned"])
    # lyrics_embedding = model.encode([matched_song["Lyrics_cleaned"]], normalize_embeddings=True)[0]

    query_embedding = combine_title_lyrics(title_embedding, lyrics_embedding, matched_song["Title_cleaned"]).reshape(1,-1)

    return {
        "status": "ok",
        "matched_song": {"song": matched_song["song"], "artist": matched_song["artist"]},
        "results": _rank_by_embedding(query_embedding, k=k, exclude_index=matched_index)
    }

if __name__ == "__main__":
    print(search_by_text("a holly song about jesus and how he sacrificed himself for me"))
    # print(search_by_text("being naughty in finding love"))
    # print(search_by_song("blanck spcace", artist="taylor swift"))


Overwriting search.py


In [12]:
%%writefile evaluation.py
import pandas as pd
from search import search_by_text
import os

def load_eval_set(path = 'evaluation/eval_queries.csv'):
    df = pd.read_csv(path, encoding='utf-8')
    eval_set = []
    for _, row in df.iterrows():
        relevant = set(t.strip() for t in row['relevant_titles'].split('|'))
        eval_set.append({'query': row['query'], 'relevant_titles': relevant})
    return eval_set

def precision_at_k(retrieved_titles, relevant_titles, k=5):
    top_k = retrieved_titles[:k]
    hits = sum(1 for t in top_k if t in relevant_titles)
    return hits / k

def recall_at_k(retrieved_titles, relevant_titles, k=5):
    top_k = retrieved_titles[:k]
    hits = sum(1 for t in top_k if t in relevant_titles)
    return hits / len(relevant_titles)

def evaluate(eval_set, k = 5, verbose = True, search_fn=search_by_text):
    precisions, recalls = [], []
    for item in eval_set:
        result = search_fn(item['query'], k=k)
        retrieved_titles = [r["song"] for r in result['results']]
        p = precision_at_k(retrieved_titles, item['relevant_titles'], k = k)
        r = recall_at_k(retrieved_titles, item['relevant_titles'], k = k)
        precisions.append(p)
        recalls.append(r)

        if verbose:
            print(f"\nQuery: {item['query']}")
            print(f"  Retrieved: {retrieved_titles}")
            print(f"  Relevant:  {item['relevant_titles']}")
            print(f"  Precision@{k}: {p:.2f}  Recall@{k}: {r:.2f}")

    mean_precision = sum(precisions) / len(precisions)
    mean_recall = sum(recalls) / len(recalls)
    print(f"\n{'='*40}\nMean Precision@{k}: {mean_precision:.3f}")
    print(f"Mean Recall@{k}: {mean_recall:.3f}\n{'='*40}")
    return {"mean_precision": mean_precision, "mean_recall": mean_recall}

if __name__ == "__main__":
    eval_set = load_eval_set()
    evaluate(eval_set, k=10, search_fn=search_by_text)


Overwriting evaluation.py


In [13]:
!python create_embeddings.py

config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 450kB/s]
README.md: 11.6kB [00:00, 22.6MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 197kB/s]
config.json: 100%|█████████████████████████████| 571/571 [00:00<00:00, 2.64MB/s]
model.safetensors: 100%|██████████████████████| 438M/438M [00:02<00:00, 167MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1406.71it/s, Materializing param=
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 363/363 [00:00<00:00, 2.44MB/s]
vocab.txt: 232kB [00:00, 10.8MB/s]
tokenizer.json: 466kB [00:00, 38.4MB/s]
config.json: 100%|██████████████████████████████| 190/190 [00:00<00:00, 804kB/s]
Enc

In [14]:
import numpy as np
song_embeddings = np.load('embeddings/song_embeddings.npy')
print(song_embeddings.shape)          # should be (num_songs, 768) now, not 384
print(np.isnan(song_embeddings).any())  # should be False

(57650, 768)
False


In [15]:
!python evaluation.py

Loading weights: 100%|█| 199/199 [00:00<00:00, 1431.28it/s, Materializing param=
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Query: fighting against my own divided personality and have conflicting different sides 
  Retrieved: ["It Doesn't Have To Be", 'Upper Me', 'Me And I', 'No Compassion', 'Threshold', 'Courage Of Your Convictions', 'State Of Art Ft. DJ Lethal', 'The Chance', 'Outside', 'Did My Time']
  Relevant:  {'Polarize'}
  Precision@10: 0.00  Recall@10: 0.00

Query: living my life based on my own choices and don't care about other opinions
  Retrieved: ['Bread For The Body', 'End Of The Line', 'Live It Up', 'Unashamed', 'Living', 'Clean Slate', 'This Is Me', 'Outro', 'The Meaning Of Life', 'My Prerogative'

In [16]:
import shutil
shutil.make_archive('/kaggle/working/embeddings_output', 'zip', '/kaggle/working/embeddings')

'/kaggle/working/embeddings_output.zip'

In [17]:
!python search.py

Loading weights: 100%|█| 199/199 [00:00<00:00, 1429.72it/s, Materializing param=
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{'status': 'ok', 'results': [{'song': 'The Giver And The Gift', 'artist': 'Point Of Grace', 'Similarity': np.float64(0.6384996377763102), 'text': "Three men and a star in the sky  \nOh what I would give for a wisdom so bright  \nOne girl with a journey to make  \nOh how I wish I was that brave  \n  \nFearless, shameless, faith through the dark  \nHow I wish I was as strong as they are  \n  \nYou say come to me wait no more  \nI give you all you're asking for  \nForget the lies this world has told  \nI'll wrap your life in linen gold  \nI'm more than just only  \nOne night that's holy  \nI?m yo